# 04 — PIO–Vehicle Matching and Exploratory Penetration / PMV

This notebook tests model-month matching between PIO sales and cleaned Wholesale/Fleet vehicle volume. It evaluates match coverage and exploratory penetration/PMV feasibility only—no forecasting, dashboard, or data export.

> **Data safety:** Clear all outputs before committing.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pio_vehicle_matching import (
    DEFAULT_MODEL_ALIASES,
    MATCH_KEYS,
    aggregate_pio_model_month,
    aggregate_pio_model_part_month,
    aggregate_vehicle_model_month,
    kpi_outlier_candidates,
    load_matching_inputs,
    match_pio_parts_to_vehicle,
    match_pio_to_vehicle,
    matching_diagnostics,
    model_code_variant_diagnostics,
    model_kpi_summary,
    monthly_pio_vehicle_summary,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load, aggregate, and match

PIO uses the step 02 loader; vehicle volume uses the cleaned step 03 output. Matching remains at model-month level using `month + tentative_brand_group + normalized_model`.

In [ ]:
inputs = load_matching_inputs(PROJECT_ROOT / "data" / "raw")
pio_model_month = aggregate_pio_model_month(inputs["pio_analysis"])
pio_model_part_month = aggregate_pio_model_part_month(inputs["pio_analysis"])
vehicle_model_month = aggregate_vehicle_model_month(inputs["vehicle_model_rows"])

matched_model_month, uniqueness = match_pio_to_vehicle(
    pio_model_month, vehicle_model_month
)
diagnostics = matching_diagnostics(matched_model_month, uniqueness)
matched_model_part = match_pio_parts_to_vehicle(
    pio_model_part_month, vehicle_model_month
)

coverage = diagnostics["coverage_summary"].iloc[0]
print(f"Workbook: {inputs['workbook_path'].name}")
print(f"PIO model-month rows: {len(pio_model_month):,}")
print(f"PIO model-part-month rows: {len(pio_model_part_month):,}")
print(f"Vehicle model-month rows: {len(vehicle_model_month):,}")

## Executive summary

The first three metrics describe key-match coverage. Positive-denominator coverage is lower and is reported separately below. High key-match coverage does not finalize variant aggregation, model-code alignment, or denominator choice.

In [ ]:
executive_summary = pd.DataFrame([
    {
        "metric": "Matched PIO model-month rows",
        "result": f"{int(coverage['matched_model_month_rows']):,} / {int(coverage['pio_model_month_rows']):,}",
    },
    {
        "metric": "Matched PIO model-month row share",
        "result": f"{coverage['matched_row_share']:.2%}",
    },
    {
        "metric": "Matched PIO quantity share",
        "result": f"{coverage['matched_quantity_share']:.4%}",
    },
    {
        "metric": "Matched PIO revenue share",
        "result": f"{coverage['matched_revenue_share']:.4%}",
    },
])
display(executive_summary)

## 2. Matching keys and compact examples

`PIS_SERI` appears to align closely with the vehicle model code. It is used below as mapping evidence, but the current merge remains exact-name based because one code can represent base, HEV, PHEV, EV, or Coupe variants. Only the explicit `GV60 → GV60 EV` alias is applied; no fuzzy matching is used.

In [ ]:
for title, table in [
    ("PIO model-month example", pio_model_month),
    ("PIO model-part-month example", pio_model_part_month),
    ("Vehicle model-month example", vehicle_model_month),
]:
    display(Markdown(f"### {title}"))
    display(table.head(5))

display(Markdown("### Tentative mapping example"))
display(
    pio_model_month[
        ["pio_brand_code", "model", "normalized_model", "tentative_brand_group"]
    ].drop_duplicates().sort_values(["tentative_brand_group", "model"]).head(5)
)
print("Explicit model aliases:", DEFAULT_MODEL_ALIASES)

## 3. Model Code / Variant Mapping Diagnostics

`PIS_SERI` is compared with the vehicle report's `Model Code` as supporting mapping evidence. Shared codes and variant-specific names demonstrate why the code should not be used alone.

In [ ]:
model_code_diagnostics = model_code_variant_diagnostics(
    inputs["pio_raw"], inputs["vehicle_model_rows"]
)

display(Markdown("### PIO models with multiple PIS_SERI values"))
display(model_code_diagnostics["pio_models_multiple_series"])

display(Markdown("### Vehicle model codes linked to multiple model names"))
display(model_code_diagnostics["vehicle_codes_multiple_models"])

display(Markdown("### Sportage code / variant example"))
sportage_variants = model_code_diagnostics["variant_candidates"].loc[
    model_code_diagnostics["variant_candidates"]["pio_model"]
    .astype("string").str.contains("Sportage", case=False, na=False)
]
display(sportage_variants)

display(Markdown("### Other likely variant-name candidates"))
display(
    model_code_diagnostics["variant_candidates"].loc[
        model_code_diagnostics["variant_candidates"]
        ["vehicle_name_has_variant_label"]
    ].head(10)
)

## 4. Match diagnostics

Coverage is separated into key matching, PIO quantity/revenue represented by those keys, and PIO observations with valid positive KPI denominators.

In [ ]:
coverage_table = diagnostics["coverage_table"].copy()
coverage_table["coverage_percent"] = coverage_table["coverage_share"] * 100
display(coverage_table[
    ["coverage_type", "basis", "covered", "total", "coverage_percent"]
])

display(Markdown("### Duplicate / many-to-many check"))
display(diagnostics["key_uniqueness"])
print("Model-part merge status:")
print(matched_model_part["_merge"].value_counts(dropna=False))

### Month and model coverage summaries

Distribution tables summarize all groups; five lowest-coverage examples identify where review should start.

In [ ]:
month_coverage = diagnostics["coverage_by_month"]
model_coverage = diagnostics["coverage_by_model"]
coverage_columns = [
    "matched_row_share", "matched_quantity_share", "matched_revenue_share",
    "valid_wholesale_quantity_share", "valid_total_quantity_share",
]

display(Markdown("**Month coverage distribution**"))
display(month_coverage[coverage_columns].describe().T)
display(Markdown("**Five lowest total-denominator month examples**"))
display(month_coverage.nsmallest(5, "valid_total_quantity_share"))

display(Markdown("**Model coverage distribution**"))
display(model_coverage[coverage_columns].describe().T)
display(Markdown("**Five lowest key-match model examples**"))
display(model_coverage.nsmallest(5, "matched_revenue_share"))

## 5. Unmatched records

Unmatched PIO records directly reduce PIO coverage. Vehicle-only rows are split into the PIO observation window versus months outside that range.

In [ ]:
display(Markdown("### Unmatched PIO summary by model"))
display(diagnostics["unmatched_pio_by_model"])
display(Markdown("**Unmatched PIO row examples**"))
display(diagnostics["unmatched_pio_model_months"].head(5))

display(Markdown("### Unmatched vehicle date-range summary"))
display(diagnostics["unmatched_vehicle_range_summary"])
display(Markdown("**Within PIO date range—five examples**"))
display(diagnostics["unmatched_vehicle_within_pio_range"].head(5))
display(Markdown("**Outside PIO date range / future—five examples**"))
display(diagnostics["unmatched_vehicle_outside_pio_range"].head(5))

## 6. Exploratory exact-name penetration and PMV definitions

- `penetration_wholesale = PIO accessory units / Wholesale vehicles`.
- `penetration_total = PIO accessory units / exploratory (Wholesale + Fleet) vehicles`.
- These are accessory-units-per-vehicle proxies, not confirmed unique-vehicle penetration rates. Because one vehicle can receive multiple accessories, values can exceed 1.
- `PMVW = PIO revenue / Wholesale vehicles`.
- `PMV_total = PIO revenue / exploratory total vehicle volume`.
- Ratios are populated only when the corresponding denominator is greater than zero. They are exploratory exact-name ratios; final variant aggregation and denominator choice require Mobis confirmation.

In [ ]:
monthly_comparison = monthly_pio_vehicle_summary(
    pio_model_month, vehicle_model_month
)
model_kpis = model_kpi_summary(matched_model_month)
ranking_eligible = model_kpis.loc[
    model_kpis["total_vehicle_volume"] >= 1_000
].copy()
penetration_outliers = kpi_outlier_candidates(
    ranking_eligible, "penetration_total"
)

display(Markdown("### Monthly comparison—five examples"))
display(monthly_comparison.head(5))
display(Markdown("### Top five models by PIO revenue"))
display(model_kpis.nlargest(5, "pio_revenue"))
display(Markdown("### Top five eligible models by penetration_total"))
display(ranking_eligible.nlargest(5, "penetration_total"))
display(Markdown("### Top five eligible models by PMV_total"))
display(ranking_eligible.nlargest(5, "PMV_total"))
display(Markdown("### Penetration outlier candidates"))
display(penetration_outliers.head(5))

## 7. Exploratory plots

In [ ]:
plot_monthly = monthly_comparison.loc[
    monthly_comparison["pio_quantity"].notna()
].copy()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, pio_column, title, color in [
    (axes[0], "pio_revenue", "Monthly PIO Revenue vs Vehicle Volume", "#D97706"),
    (axes[1], "pio_quantity", "Monthly PIO Quantity vs Vehicle Volume", "#2E7D32"),
]:
    vehicle_axis = ax.twinx()
    ax.plot(plot_monthly["month"], plot_monthly[pio_column], marker="o", color=color)
    vehicle_axis.plot(
        plot_monthly["month"], plot_monthly["total_vehicle_volume"],
        marker="s", color="#0066A1",
    )
    ax.set_title(title)
    ax.set_ylabel(pio_column.replace("_", " "))
    vehicle_axis.set_ylabel("exploratory vehicle volume")
axes[1].set_xlabel("Month")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

top_penetration = ranking_eligible.nlargest(10, "penetration_total").sort_values("penetration_total")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_penetration["model"].astype("string"), top_penetration["penetration_total"], color="#7E57C2")
ax.set_title("Top Eligible Models by Exploratory Total Penetration")
ax.set_xlabel("accessory units per exploratory total vehicle")
fig.tight_layout()
plt.show()

## 8. Validation

In [ ]:
assert not uniqueness["many_to_many_warning"].any()
assert not pio_model_month.duplicated(MATCH_KEYS).any()
assert not vehicle_model_month.duplicated(MATCH_KEYS).any()
assert len(matched_model_part) == len(pio_model_part_month)
assert len(matched_model_month) <= len(pio_model_month) + len(vehicle_model_month)
assert matched_model_month.loc[
    matched_model_month["penetration_wholesale"].notna(), "wholesale_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["penetration_total"].notna(), "total_vehicle_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["PMVW"].notna(), "wholesale_volume"
].gt(0).all()
assert matched_model_month.loc[
    matched_model_month["PMV_total"].notna(), "total_vehicle_volume"
].gt(0).all()
print("All matching and denominator checks passed.")

## 9. Conclusions

### Working assumptions

- K maps to KUS; Genesis-like H models map to GMA; other H models map to HMA.
- `GV60 → GV60 EV` is the only explicit alias. Current matching uses month, normalized model name, and tentative brand group; `PIS_SERI` is retained as validation evidence rather than used alone.
- Exploratory total volume requires both Wholesale and Fleet values at model-month level.

### Key findings

- Key-match coverage is very high, but valid denominator coverage is lower and must be evaluated separately.
- High key-match coverage does not mean the denominator definition is finalized. Variant aggregation and model-code alignment still require sponsor confirmation before penetration or PMV can be treated as final business KPIs.
- PIO `Rio` and `Stinger` are unmatched in the current vehicle report.
- Vehicle-only rows include unmatched variants within the PIO window plus future months outside it.
- Penetration values are accessory-units-per-vehicle proxies and can exceed 1.

### Questions for Mobis

1. Confirm H/K to HMA/GMA/KUS mapping and official model aliases.
2. Confirm whether blank Wholesale/Fleet cells mean zero, unavailable, or future data.
3. Confirm whether the final denominator should be Wholesale, Fleet, total volume, or another measure.
4. Confirm whether PIO quantity counts accessory units or vehicles with accessories.
5. Confirm the official definitions of PMV and PMVW.

**Stopping point:** no forecasting model, dashboard, CSV, or processed dataset is created.